# 08 更新数据

日常回测不用运行此文件；总入口只读取现有 CSV。需要更新时，先在项目解释器安装 `yfinance`，再将下面 `DOWNLOAD` 改为 `True` 并 Run All。默认保存为 `data/raw/etf_daily_download.csv`，便于检查后再替换原始数据。下载标的、起始日及 `auto_adjust=False` 与原代码一致。


In [1]:
from pathlib import Path
import os
import json

def find_pipeline_dir():
    """支持从项目根目录、pipeline 目录及项目内子目录启动。"""
    roots = ([Path(os.environ['ETF_PROJECT_ROOT']).expanduser()] if os.environ.get('ETF_PROJECT_ROOT') else [])
    roots += [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for candidate in (root / 'pipeline', root):
            if (candidate / '00_engine.ipynb').is_file() and (candidate / '07_run_pipeline.ipynb').is_file():
                return candidate.resolve()
    raise FileNotFoundError('找不到 pipeline 文件夹。请将 PyCharm 工作目录设为项目根目录，或设置 ETF_PROJECT_ROOT。')

PIPELINE_DIR = find_pipeline_dir()

def load_notebook_definitions(filename):
    """独立命名空间加载函数；不执行被加载 notebook 的入口单元格。"""
    path = PIPELINE_DIR / filename
    notebook = json.loads(path.read_text(encoding='utf-8'))
    namespace = {'__name__': 'pipeline_module'}
    for number, item in enumerate(notebook['cells']):
        if item['cell_type'] == 'code' and 'pipeline-entry' not in item.get('metadata', {}).get('tags', []):
            exec(compile(''.join(item['source']), f'{path}:cell{number}', 'exec'), namespace)
    return namespace

_engine = load_notebook_definitions('00_engine.ipynb')
for _name, _value in _engine.items():
    if not _name.startswith('_'):
        globals()[_name] = _value


In [ ]:
def download_data(output_path, start='2011-10-01'):
    import yfinance as yf
    data = yf.download(TICKS + ['ES3.SI', 'SGD=X'], start=start, auto_adjust=False)
    if data.empty: raise ValueError('下载结果为空，未保存。')
    missing = [ticker for ticker in TICKS if ('Adj Close', ticker) not in data or data['Adj Close'][ticker].dropna().empty]
    if missing: raise ValueError(f'下载缺少标的：{missing}；未保存。')
    path = Path(output_path).expanduser().resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(path)
    print('已保存:', path, '| 最后日期:', data.index[-1].date())
    return data

def main(download=False, output_path=None):
    if not download:
        print('未下载。需要更新时，将 DOWNLOAD 改为 True。')
        return None
    output_path = output_path or PIPELINE_DIR.parent / 'data' / 'raw' / 'etf_daily_download.csv'
    return download_data(output_path)


In [3]:
DOWNLOAD = False
OUTPUT_PATH = None
results = main(download=DOWNLOAD, output_path=OUTPUT_PATH)


未下载。需要更新时，将 DOWNLOAD 改为 True。
